In [7]:
# automatically reloads all modules before executing a new cell
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import sys
sys.path.insert(0, '/home/mariia/AstroML/')
sys.path.insert(0, '/home/mariia/AstroML/src/')

In [83]:
import os
import torch
from torch.utils.data import DataLoader
import wandb
from datasets import load_dataset
import numpy as np
from collections import defaultdict
import json
from scipy.stats import ttest_rel
from src.dataset import HFPSMDataset

In [123]:
ds = load_dataset('MeriDK/AstroM3Dataset', name='sub10_42', split='train')

In [124]:
el = ds[0]

In [127]:
len(el['photometry'])

192

In [129]:
len(el['spectra'])

3909

In [7]:
def ddict():
    return defaultdict(ddict)

res = ddict()

In [8]:
ids = ddict()
api = wandb.Api()
runs = api.runs('AstroM3')

for run in runs:
    mode = run.config['mode']
    pretrain = True if run.config['use_pretrain'] is True else False
    sub = run.config['data_sub']
    seed = run.config['random_seed']
    ids[mode][pretrain][sub][seed] = run.id

In [14]:
for mode in ('spectra', 'meta', 'photo', 'all'):
    print(f"'{mode}': {{")
    for pretrain in (False, True):
        print(f"\t{pretrain}: {{")
        for sub in ('sub10', 'sub25', 'sub50', 'full'):
            print(f"\t\t'{sub}': {{", end='')
            for seed in (42, 0, 66, 12):
                print(f"{seed}: '{ids[mode][pretrain][sub][seed]}', ", end='')
            print(f"{123}: '{ids[mode][pretrain][sub][123]}'}}," )
        print("\t},")
    print("},")

'spectra': {
	False: {
		'sub10': {42: 'jx5u1ynn', 0: 'o7ouxm7r', 66: 'o9josicx', 12: '4sf41lk2', 123: '16cey292'},
		'sub25': {42: 'rzx4dh71', 0: 'yd1na38z', 66: 'nto5uvs0', 12: 'py8c95cx', 123: 'p3ec2bdk'},
		'sub50': {42: '56979zxw', 0: '66b5ui6r', 66: '8e34m91t', 12: 'mjnc2b8z', 123: '1iuysc4s'},
		'full': {42: 'bjn836x0', 0: 'nmtdyve6', 66: 'vh85spjt', 12: '8w4hrn1q', 123: '5nyk8b93'},
	},
	True: {
		'sub10': {42: 'gr1kgco1', 0: 'mh837xql', 66: 'xg66aech', 12: 'xxw1jffs', 123: 'nwtp6yst'},
		'sub25': {42: '5sgu488f', 0: 'ns0scmtf', 66: 'ra4zobzu', 12: 'y4u681fc', 123: 'pysl0zra'},
		'sub50': {42: '2t6hw3b4', 0: 'db5ldmjy', 66: 'e158eh1h', 12: 'cus5xgfn', 123: 'a1dmvdty'},
		'full': {42: '7duigpu6', 0: 'gurys2pl', 66: 'cbhn5crj', 12: 'ae0xnmfq', 123: 'rint4ydd'},
	},
},
'meta': {
	False: {
		'sub10': {42: 'fpjfk551', 0: 'd3e31vjn', 66: '585g4my3', 12: '85j8j6de', 123: 'ywn0sm8n'},
		'sub25': {42: 'u5vseua7', 0: '5fbphtzv', 66: 'xj98adhf', 12: 'lupductk', 123: 'p7shnlkp'},
		'sub50'

In [104]:
def print_res(resutls, mode):
    print(mode)
    for sub in ('sub10', 'sub25', 'sub50', 'full'):
        acc = list(results[mode]['False'][sub].values())
        acc_clip  = list(results[mode]['True'][sub].values())
        
        mean = np.mean(acc) * 100
        std = np.std(acc) * 100

        mean_clip = np.mean(acc_clip) * 100
        std_clip = np.std(acc_clip) * 100

        p_value = ttest_rel(acc, acc_clip).pvalue

        print(f"{sub:<6} {'False':<5} {round(mean, 3):>6} ± {round(std, 3):<6}")
        print(f"{'':<6} {'True':<5} {round(mean_clip, 3):>6} ± {round(std_clip, 3):<6} {round(p_value, 6):<8} {'+' if p_value < 0.05 else '-'}")
        print(f"{'':<6} {'Diff':<5} {round(mean_clip - mean, 2):>6} from {round(mean, 2):>6} to {round(mean_clip, 2):>6}")

In [130]:
with open('/home/mariia/AstroML/results.json', 'r') as f:
    results = json.load(f)

In [85]:
noclip = list(results['spectra']['False']['sub25'].values())
clip = list(results['spectra']['True']['sub25'].values())

In [88]:
ttest_rel(noclip, clip)

TtestResult(statistic=-1.913537272629284, pvalue=0.12822227105087375, df=4)

In [140]:
results['all']['True']['full']

{'42': 0.9366292134831461,
 '0': 0.9232142857142858,
 '66': 0.946284691136974,
 '12': 0.9354549529359032,
 '123': 0.9334821428571428}

In [105]:
print_res(results, 'spectra')

spectra
sub10  False 46.677 ± 3.486 
       True  60.963 ± 2.584  0.002294 +
       Diff   14.29 from  46.68 to  60.96
sub25  False 63.729 ± 1.637 
       True  66.676 ± 2.549  0.128222 -
       Diff    2.95 from  63.73 to  66.68
sub50  False 68.072 ± 1.759 
       True  71.376 ± 1.465  0.005887 +
       Diff     3.3 from  68.07 to  71.38
full   False 76.278 ± 0.931 
       True  77.056 ± 1.033  0.17183  -
       Diff    0.78 from  76.28 to  77.06


In [106]:
print_res(results, 'meta')

meta
sub10  False 74.888 ± 1.856 
       True  77.161 ± 1.476  0.010251 +
       Diff    2.27 from  74.89 to  77.16
sub25  False 78.933 ± 2.419 
       True  81.667 ± 1.332  0.040461 +
       Diff    2.73 from  78.93 to  81.67
sub50  False 82.628 ± 1.048 
       True  83.076 ± 1.367  0.168866 -
       Diff    0.45 from  82.63 to  83.08
full   False 85.667 ± 0.858 
       True  85.864 ± 0.739  0.431235 -
       Diff     0.2 from  85.67 to  85.86


In [134]:
print_res(results, 'photo')

photo
sub10  False 84.527 ± 2.655 
       True  91.264 ± 0.974  0.007899 +
       Diff    6.74 from  84.53 to  91.26
sub25  False 78.382 ± 2.359 
       True   88.58 ± 0.948  0.001474 +
       Diff    10.2 from  78.38 to  88.58
sub50  False 85.197 ± 6.209 
       True  89.572 ± 0.652  0.240576 -
       Diff    4.38 from   85.2 to  89.57
full   False 90.724 ± 1.464 
       True  91.101 ± 0.589  0.610913 -
       Diff    0.38 from  90.72 to   91.1


In [108]:
print_res(results, 'all')

all
sub10  False 88.261 ± 1.248 
       True    89.9 ± 2.0    0.120962 -
       Diff    1.64 from  88.26 to   89.9
sub25  False 90.811 ± 1.03  
       True  91.243 ± 1.254  0.185437 -
       Diff    0.43 from  90.81 to  91.24
sub50  False 91.977 ± 0.272 
       True  92.174 ± 0.719  0.659748 -
       Diff     0.2 from  91.98 to  92.17
full   False 94.127 ± 0.24  
       True  93.501 ± 0.737  0.097018 -
       Diff   -0.63 from  94.13 to   93.5


In [125]:
acc = list(results['spectra']['False']['sub10'].values())
np.mean(acc)

0.47679119966791206

In [130]:
acc = list(results['spectra']['True']['sub10'].values())
np.mean(acc)

0.5805022831050228

In [126]:
acc = list(results['spectra']['False']['sub25'].values())
np.mean(acc)

0.6124463829868595

In [131]:
acc = list(results['spectra']['True']['sub25'].values())
np.mean(acc)

0.6768326500999268

In [127]:
acc = list(results['spectra']['False']['sub50'].values())
np.mean(acc)

0.6744481265628313

In [143]:
for sub in ('sub10', 'sub25', 'sub50', 'full'):
    for clip in ('False', 'True'):
        acc = list(results['photo'][clip][sub].values())
        print(f"{sub:<6} {str(clip):<5} {round(np.mean(acc) * 100, 3):>6} ± {round(np.std(acc) * 100, 3):<6}")

sub10  False 82.344 ± 2.096 
sub10  True  90.265 ± 0.727 
sub25  False 79.154 ± 3.545 
sub25  True  88.109 ± 1.292 
sub50  False 82.978 ± 8.601 
sub50  True   89.86 ± 0.475 
full   False 91.057 ± 0.832 
full   True  91.029 ± 0.814 


In [141]:
for sub in ('sub10', 'sub25', 'sub50', 'full'):
    for clip in ('False', 'True'):
        acc = list(results['meta'][clip][sub].values())
        print(f"{sub:<6} {str(clip):<5} {round(np.mean(acc) * 100, 3):>6} ± {round(np.std(acc) * 100, 3):<6}")

sub10  False 75.703 ± 1.888 
sub10  True    77.8 ± 1.591 
sub25  False 78.898 ± 2.807 
sub25  True  81.559 ± 0.915 
sub50  False 82.771 ± 1.214 
sub50  True  83.346 ± 1.375 
full   False 85.596 ± 0.716 
full   True   85.91 ± 0.646 


In [144]:
for sub in ('sub10', 'sub25', 'sub50', 'full'):
    for clip in ('False', 'True'):
        acc = list(results['spectra'][clip][sub].values())
        print(f"{sub:<6} {str(clip):<5} {round(np.mean(acc) * 100, 3):>6} ± {round(np.std(acc) * 100, 3):<6}")

sub10  False 47.679 ± 4.478 
sub10  True   58.05 ± 3.163 
sub25  False 61.245 ± 4.94  
sub25  True  67.683 ± 1.327 
sub50  False 67.445 ± 0.958 
sub50  True  71.357 ± 0.938 
full   False 75.974 ± 1.417 
full   True  77.369 ± 0.925 


In [ ]:
acc = list(results['spectra']['False']['sub10'].values())
np.mean(acc)

In [3]:
def to_ddict(d):
    """Recursively convert a dict to a defaultdict."""
    if isinstance(d, dict):
        return defaultdict(ddict, {k: to_ddict(v) for k, v in d.items()})
    return d

In [96]:
results = to_ddict(results)

In [97]:
results['meta']['true']['sub10']['12'] = 0.877

In [98]:
results

defaultdict(<function __main__.ddict()>,
            {'photo': defaultdict(<function __main__.ddict()>,
                         {'false': defaultdict(<function __main__.ddict()>,
                                      {'full': defaultdict(<function __main__.ddict()>,
                                                   {'123': 0.9111607142857143})})}),
             'meta': defaultdict(<function __main__.ddict()>,
                         {'true': defaultdict(<function __main__.ddict()>,
                                      {'sub10': defaultdict(<function __main__.ddict()>,
                                                   {'12': 0.877})})})})

In [99]:
with open('/home/mariia/AstroML/results2.json', 'w') as f:
    json.dump(results, f, indent=4)

In [100]:
with open('/home/mariia/AstroML/results2.json', 'r') as f:
    results = json.load(f)

In [101]:
results

{'photo': {'false': {'full': {'123': 0.9111607142857143}}},
 'meta': {'true': {'sub10': {'12': 0.877}}}}

In [44]:
def eval_run(run):
    config = run.config
    config['use_wandb'] = False
    
    ds = load_dataset('MeriDK/AstroM3Processed', name=f'{config["data_sub"]}_{config["random_seed"]}_norm')
    test_dataset = HFPSMDataset(ds, classes=config['classes'], seq_len=config['seq_len'], split='test')
    test_dataloader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
    
    criterion = torch.nn.CrossEntropyLoss()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = get_model(config)
    model = model.to(device)
    weights_path = os.path.join('/home/mariia/AstroML', config['weights_path'] + '-' + run.id, 'weights-best.pth')
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    
    trainer = Trainer(model=model, optimizer=None, scheduler=None, warmup_scheduler=None, criterion=criterion, device=device, config=config)
    acc = trainer.evaluate(test_dataloader, test_dataset.id2label, plot=False)

    return acc

In [11]:
def ddict():
    return defaultdict(ddict)

res = ddict()

In [40]:
api = wandb.Api()
runs = api.runs('AstroM3')

In [5]:
r = runs[0]
r.id, r.config['data_sub'], r.config['weights_path'], r.config['random_seed'], r.config['mode'], r.state == 'finished', 

('ern9rvl5', 'full', './weights/2025-02-20-16-54', 42, 'clip', True)

In [47]:
for mode in ('spectra', 'photo', 'meta', 'all'):
    for pretrain in (True, False):
        for sub in ('full', 'sub50', 'sub25', 'sub10'):
            print(mode, pretrain, sub)
            seeds = []
            
            for run in runs:
                c = run.config
                c_pretrain = True if c['use_pretrain'] is True else False
                
                if c['mode'] == mode and c_pretrain == pretrain and c['data_sub'] == sub:
                    seeds.append(c['random_seed'])

            if len(seeds) != 5 and set(seeds) != set([42, 0, 12, 123, 66]):
                print('Oooops')

spectra True full
spectra True sub50
spectra True sub25
spectra True sub10
spectra False full
spectra False sub50
spectra False sub25
spectra False sub10
photo True full
photo True sub50
photo True sub25
photo True sub10
photo False full
photo False sub50
photo False sub25
photo False sub10
meta True full
meta True sub50
meta True sub25
meta True sub10
meta False full
meta False sub50
meta False sub25
meta False sub10
all True full
all True sub50
all True sub25
all True sub10
all False full
all False sub50
all False sub25
all False sub10


In [55]:
ids = ddict()

for run in runs:
    mode = run.config['mode']
    pretrain = True if run.config['use_pretrain'] is True else False
    sub = run.config['data_sub']
    seed = run.config['random_seed']
    ids[mode][pretrain][sub][seed] = run.id

In [59]:
def defaultdict_to_dict(d):
    if isinstance(d, defaultdict):
        return {k: defaultdict_to_dict(v) for k, v in d.items()}
    return d

In [83]:
results = ddict()

In [84]:
results['spectra'][True]['sub10'][0] = 0.75

In [85]:
results

defaultdict(<function __main__.ddict()>,
            {'spectra': defaultdict(<function __main__.ddict()>,
                         {True: defaultdict(<function __main__.ddict()>,
                                      {'sub10': defaultdict(<function __main__.ddict()>,
                                                   {0: 0.75})})})})

In [86]:
with open('/home/mariia/AstroML/results.json', 'w') as f:
    json.dump(results, f, indent=4)

In [87]:
with open('/home/mariia/AstroML/results.json', 'r') as f:
    results = json.load(f)

In [88]:
results

{'spectra': {'true': {'sub10': {'0': 0.75}}}}

In [81]:
with open('/home/mariia/AstroML/results.json', 'r') as f:
    results.update(json.load(f))

In [82]:
results

defaultdict(<function __main__.ddict()>,
            {'spectra': {'true': {'sub10': {'0': 0.5636363636363636}}}})

In [76]:
def pretty_print(data):
    def format_dict(d, indent=0):
        for key, value in d.items():
            if isinstance(value, dict):
                print(" " * indent + f"{repr(key)}: {{")
                format_dict(value, indent + 4)
                print(" " * indent + "},")
            else:
                print(f"{repr(key)}: {json.dumps(value)},", end='')  # Ensures IDs are in one line

    print("{")
    format_dict(data, indent=4)
    print("}")

In [78]:
res = {
    'spectra': {
        False: {
            'sub10': {42: "gxfnczum", 0: "lv5otwsp", 66: "w4hjnpbs", 123: "8q6btqwm", 12: "yvp1nigg"},
            'sub25': {42: "rlc5b2v5", 0: "oitqcxxy", 66: "zdnr9wwt", 123: "we2w8uu5", 12: "foqd708r"},
            'sub50': {42: "zinjol72", 0: "074a189n", 66: "6h4w0us5", 12: "70ko2c3k", 123: "qzioohea"},
            'full': {42: "o8y9hf99", 0: "2jlq2wnl", 66: "hofvrjj0", 12: "6ejqjoac", 123: "94d129rw"}
        },
        True: {
            'sub10': {66: "giyv3cfh", 12: "u36t4n6b", 0: "dl0p0j93", 123: "xznyxtlh", 42: "dpb1yfl3"},
            'sub25': {42: "xet37dgx", 0: "5y7ckjlo", 66: "kq098os4", 12: "iubaeo5g", 123: "yvw984ei"},
            'sub50': {12: "cz1doijc", 66: "vnv9a6ig", 0: "73bwv7b2", 123: "7pkrv4m8", 42: "zt50o9p1"},
            'full': {42: "6xwvxl7m", 0: "p2l77xhw", 12: "xqyfe8nf", 66: "i05lzbpo", 123: "usksi654"}
        }
    },
    'meta': {
        False: {
            'sub10': {42: "t0hjwfnm", 0: "b1m2lrt4", 66: "yb7iyzkv", 12: "oaqaq0c4", 123: "dyzehjmr"},
            'sub25': {42: "t7jlisie", 0: "cmdgsfd8", 66: "hxwdvwyq", 12: "8x4qhvec", 123: "uh0evyir"},
            'sub50': {42: "rpxg8xl0", 0: "e50t40ah", 66: "ulubt2fh", 12: "tyeb3eq4", 123: "etu4niij"},
            'full': {42: "4l7fdezl", 0: "ll7adzjm", 66: "tkgg6kpq", 12: "mwuvq3p7", 123: "8bpf56or"}
        },
        True: {
            'sub10': {42: "0ryv7wng", 123: "3418js9l", 66: "1pb0eg6c", 12: "8mw4n79c", 0: "rmo9gu89"},
            'sub25': {42: "2uuo8snv", 123: "voy37o0m", 66: "wpdtq4aq", 12: "o2nv8zx1", 0: "fnp6969w"},
            'sub50': {42: "3uehfui4", 123: "2sapvqfu", 66: "ewiqvto8", 0: "ensfo1yf", 12: "0b5j63dy"},
            'full': {42: "j6y34hyv", 12: "dmae6vl1", 66: "jqnqpey8", 0: "06cd7pr6", 123: "oj6ovpw2"}
        }
    },
    'photo': {
        False: {
            'sub10': {42: "bhhbredp", 0: "uvntusjc", 66: "pia3tb4m", 12: "o3erl0ca", 123: "ka7puer5"},
            'sub25': {42: "8wtmsigk", 0: "fr9os1u2", 66: "w9t64rum", 12: "kal596d6", 123: "aslof8mn"},
            'sub50': {42: "hz0wwyni", 0: "tgmssy11", 66: "wxiydt82", 12: "fcth4u8u", 123: "3x93x6wh"},
            'full': {42: "s1ydei13", 0: "qxslbup4", 66: "3ysrf9fc", 12: "3f7pgmvt", 123: "u80ib6m7"}
        },
        True: {
            'sub10': {42: "gh89cwc6", 0: "hlxt7816", 66: "2lj4bdbn", 123: "5zk8dzc1", 12: "txafnzza"},
            'sub25': {42: "i6pun92h", 0: "m3n273ei", 12: "xx3cyomr", 66: "9oanyi02", 123: "zw7tt41v"},
            'sub50': {42: "tn9rq3t2", 0: "6mg2v78y", 66: "4xb5huf7", 12: "4nb5adeu", 123: "9kdw8hra"},
            'full': {42: "4x2dh9gn", 0: "nlfpjk7k", 12: "seq6z71u", 66: "9wq9887a", 123: "h61zyelv"}
        }
    },
    'all': {
        False: {
            'sub10': {42: "fpjnkj8x", 0: "oh2jdiv0", 66: "853z069w", 12: "0niorzpw", 123: "4g4oh8z6"},
            'sub25': {42: "3n3ncjae", 0: "ob3lj01e", 66: "ijwkly5q", 12: "fxg8j2xn", 123: "qd8woip5"},
            'sub50': {42: "1ag0z3qj", 0: "auy22jr2", 66: "3xmt9tn7", 12: "hkahenqy", 123: "dv49b6m4"},
            'full': {42: "osm8geo2", 0: "k6zoo5vi", 66: "85u0mgb7", 12: "4d75anka", 123: "fhcaspw7"}
        },
        True: {
            'sub10': {12: "fyo07o2x", 0: "r6gurgvb", 66: "ib5anpqv", 42: "dlb68mtz", 123: "jsyeeshv"},
            'sub25': {12: "a9j76o9m", 0: "t0rln1pu", 66: "0bze25m3", 42: "mde8nwfn", 123: "ndrcmilk"},
            'sub50': {12: "3mfhzic8", 0: "93ukwf6i", 66: "mupv0ne9", 42: "rncxme7d", 123: "4opc8szp"},
            'full': {0: "gw7eieec", 66: "cxsn0uye", 12: "kdlom7f6", 42: "ltfabnm9", 123: "gp62c77y"}
        }
    }
}

In [104]:
run_ids2 = {
    'spectra': {
        False: {
            'sub10': {42: 'gxfnczum', 0: 'lv5otwsp', 66: 'w4hjnpbs', 12: 'yvp1nigg', 123: '8q6btqwm'},
            'sub25': {42: 'rlc5b2v5', 0: 'oitqcxxy', 66: 'zdnr9wwt', 12: 'foqd708r', 123: 'we2w8uu5'},
            'sub50': {42: 'zinjol72', 0: '074a189n', 66: '6h4w0us5', 12: '70ko2c3k', 123: 'qzioohea'},
            'full': {42: 'o8y9hf99', 0: '2jlq2wnl', 66: 'hofvrjj0', 12: '6ejqjoac', 123: '94d129rw'}
        },
        True: {
            'sub10': {42: 'dpb1yfl3', 0: 'dl0p0j93', 66: 'giyv3cfh', 12: 'u36t4n6b', 123: 'xznyxtlh'},
            'sub25': {42: 'xet37dgx', 0: '5y7ckjlo', 66: 'kq098os4', 12: 'iubaeo5g', 123: 'yvw984ei'},
            'sub50': {42: 'zt50o9p1', 0: '73bwv7b2', 66: 'vnv9a6ig', 12: 'cz1doijc', 123: '7pkrv4m8'},
            'full': {42: '6xwvxl7m', 0: 'p2l77xhw', 66: 'i05lzbpo', 12: 'xqyfe8nf', 123: 'usksi654'}
        }
    },
    'meta': {
        False: {
            'sub10': {42: 't0hjwfnm', 0: 'b1m2lrt4', 66: 'yb7iyzkv', 12: 'oaqaq0c4', 123: 'dyzehjmr'},
            'sub25': {42: 't7jlisie', 0: 'cmdgsfd8', 66: 'hxwdvwyq', 12: '8x4qhvec', 123: 'uh0evyir'},
            'sub50': {42: 'rpxg8xl0', 0: 'e50t40ah', 66: 'ulubt2fh', 12: 'tyeb3eq4', 123: 'etu4niij'},
            'full': {42: '4l7fdezl', 0: 'll7adzjm', 66: 'tkgg6kpq', 12: 'mwuvq3p7', 123: '8bpf56or'}
        },
        True: {
            'sub10': {42: '0ryv7wng', 0: 'rmo9gu89', 66: '1pb0eg6c', 12: '8mw4n79c', 123: '3418js9l'},
            'sub25': {42: '2uuo8snv', 0: 'fnp6969w', 66: 'wpdtq4aq', 12: 'o2nv8zx1', 123: 'voy37o0m'},
            'sub50': {42: '3uehfui4', 0: 'ensfo1yf', 66: 'ewiqvto8', 12: '0b5j63dy', 123: '2sapvqfu'},
            'full': {42: 'j6y34hyv', 0: '06cd7pr6', 66: 'jqnqpey8', 12: 'dmae6vl1', 123: 'oj6ovpw2'}
        }
    },
    'photo': {
        False: {
            'sub10': {42: 'bhhbredp', 0: 'uvntusjc', 66: 'pia3tb4m', 12: 'o3erl0ca', 123: 'ka7puer5'},
            'sub25': {42: '8wtmsigk', 0: 'fr9os1u2', 66: 'w9t64rum', 12: 'kal596d6', 123: 'aslof8mn'},
            'sub50': {42: 'hz0wwyni', 0: 'tgmssy11', 66: 'wxiydt82', 12: 'fcth4u8u', 123: '3x93x6wh'},
            'full': {42: 's1ydei13', 0: 'qxslbup4', 66: '3ysrf9fc', 12: '3f7pgmvt', 123: 'u80ib6m7'}
        },
        True: {
            'sub10': {42: 'gh89cwc6', 0: 'hlxt7816', 66: '2lj4bdbn', 12: 'txafnzza', 123: '5zk8dzc1'},
            'sub25': {42: 'i6pun92h', 0: 'm3n273ei', 66: '9oanyi02', 12: 'xx3cyomr', 123: 'zw7tt41v'},
            'sub50': {42: 'tn9rq3t2', 0: '6mg2v78y', 66: '4xb5huf7', 12: '4nb5adeu', 123: '9kdw8hra'},
            'full': {42: '4x2dh9gn', 0: 'nlfpjk7k', 66: '9wq9887a', 12: 'seq6z71u', 123: 'h61zyelv'}
        }
    },
    'all': {
        False: {
            'sub10': {42: 'fpjnkj8x', 0: 'oh2jdiv0', 66: '853z069w', 12: '0niorzpw', 123: '4g4oh8z6'},
            'sub25': {42: '3n3ncjae', 0: 'ob3lj01e', 66: 'ijwkly5q', 12: 'fxg8j2xn', 123: 'qd8woip5'},
            'sub50': {42: '1ag0z3qj', 0: 'auy22jr2', 66: '3xmt9tn7', 12: 'hkahenqy', 123: 'dv49b6m4'},
            'full': {42: 'osm8geo2', 0: 'k6zoo5vi', 66: '85u0mgb7', 12: '4d75anka', 123: 'fhcaspw7'}
        },
        True: {
            'sub10': {42: 'dlb68mtz', 0: 'r6gurgvb', 66: 'ib5anpqv', 12: 'fyo07o2x', 123: 'jsyeeshv'},
            'sub25': {42: 'mde8nwfn', 0: 't0rln1pu', 66: '0bze25m3', 12: 'a9j76o9m', 123: 'ndrcmilk'},
            'sub50': {42: 'rncxme7d', 0: '93ukwf6i', 66: 'mupv0ne9', 12: '3mfhzic8', 123: '4opc8szp'},
            'full': {42: 'ltfabnm9', 0: 'gw7eieec', 66: 'cxsn0uye', 12: 'kdlom7f6', 123: 'gp62c77y'}
        }
    }
}

In [105]:
for mode in ('spectra', 'photo', 'meta', 'all'):
    for pretrain in (True, False):
        for sub in ('full', 'sub50', 'sub25', 'sub10'):
            for seed in (42, 0, 66, 12, 123):
                if res[mode][pretrain][sub][seed] != run_ids2[mode][pretrain][sub][seed]:
                    print(mode, pretrain, sub, seed)

In [18]:
mode = 'meta'

for sub in ('full', 'sub50', 'sub25', 'sub10'):
    for run in runs:
        if run.config['mode'] == mode and run.config['data_sub'] == sub:
            random_seed = run.config['random_seed']
            use_pretrain = True if run.config['use_pretrain'] is True else False
            
            print(mode, sub, random_seed, use_pretrain)
            try:
                res[mode][use_pretrain][sub][random_seed] = eval_run(run)
            except Exception as e:
                print(f"Error processing {mode}, {sub}, seed {random_seed}, pretrain {use_pretrain}: {e}")

meta full 42 False


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:34<00:00, 18.83s/it]


meta full 0 False


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:47<00:00, 21.45s/it]


meta full 66 False


...

meta sub10 12 False


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:07<00:00,  8.00s/it]


meta sub10 123 False


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:08<00:00,  8.28s/it]


meta sub10 42 True
Error processing meta, sub10, seed 42, pretrain True: [Errno 2] No such file or directory: './weights/2025-02-20-16-54-ern9rvl5/weights-best.pth'
meta sub10 123 True
Error processing meta, sub10, seed 123, pretrain True: [Errno 2] No such file or directory: './weights/2025-02-21-10-37-i48fhd95/weights-best.pth'
meta sub10 66 True
Error processing meta, sub10, seed 66, pretrain True: [Errno 2] No such file or directory: './weights/2025-02-21-07-01-yvynovaz/weights-best.pth'
meta sub10 12 True
Error processing meta, sub10, seed 12, pretrain True: [Errno 2] No such file or directory: './weights/2025-02-21-10-37-vzgc136n/weights-best.pth'
meta sub10 0 True
Error processing meta, sub10, seed 0, pretrain True: [Errno 2] No such file or directory: './weights/2025-02-20-21-34-yjofdq2i/weights-best.pth'


In [43]:
if res['spectra'][]:
    print('yes')

In [19]:
mode = 'meta'

for sub in ('full', 'sub50', 'sub25', 'sub10'):
    for run in runs:
        if run.config['mode'] == mode and run.config['data_sub'] == sub and run.config['use_pretrain'] is True:
            random_seed = run.config['random_seed']
            use_pretrain = True if run.config['use_pretrain'] is True else False
            print(mode, sub, random_seed, use_pretrain)
            
            try:
                if res[mode][use_pretrain][sub][random_seed]:
                    print(f'Duplicate at {mode} {use_pretrain} {sub} {random_seed}')
                res[mode][use_pretrain][sub][random_seed] = eval_run(run)
            except Exception as e:
                print(f"Error processing {mode}, {sub}, seed {random_seed}, pretrain {use_pretrain}: {e}")

meta full 42 True


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:27<00:00, 17.59s/it]


meta full 12 True


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:27<00:00, 17.42s/it]


meta full 66 True


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:09<00:00,  9.28s/it]


meta sub10 12 True


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:07<00:00,  7.45s/it]


meta sub10 0 True


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:07<00:00,  7.13s/it]


In [27]:
for sub in ('sub10', 'sub25', 'sub50', 'full'):
    for clip in (True, False):
        acc = list(res['meta'][clip][sub].values())
        print(f"{sub:<6} {str(clip):<5} {round(np.mean(acc) * 100, 3):>6} ± {round(np.std(acc) * 100, 3):<6}")

sub10  True    77.8 ± 1.591 
sub10  False 75.703 ± 1.888 
sub25  True  81.559 ± 0.915 
sub25  False 78.898 ± 2.807 
sub50  True  83.346 ± 1.375 
sub50  False 82.771 ± 1.214 
full   True  86.008 ± 0.516 
full   False 85.596 ± 0.716 


In [ ]:
mode = 'photo'

for sub in ('full', 'sub50', 'sub25', 'sub10'):
    for run in runs:
        if run.config['mode'] == mode and run.config['data_sub'] == sub:
            random_seed = run.config['random_seed']
            use_pretrain = True if run.config['use_pretrain'] is True else False
            
            print(mode, sub, random_seed, use_pretrain)
            try:
                res[mode][use_pretrain][sub][random_seed] = eval_run(run)
            except Exception as e:
                print(f"Error processing {mode}, {sub}, seed {random_seed}, pretrain {use_pretrain}: {e}")

photo full 42 False


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:28<00:00, 17.75s/it]


photo full 0 False


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:42<00:00, 20.59s/it]


photo full 66 False


...

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:31<00:00, 18.33s/it]


photo full 12 True


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:27<00:00, 17.49s/it]


photo full 66 True


 20%|████████████████▊                                                                   | 1/5 [00:16<01:06, 16.50s/it]

In [31]:
for mode in ('photo', 'all'):
    for data_sub in ('sub50', 'sub25', 'sub10'):
        for run in runs:
            if run.config['mode'] == mode and run.config['data_sub'] == data_sub and not run.config['use_pretrain']:
                print(mode, data_sub, run.config['random_seed'])
                res[mode][data_sub][run.config['random_seed']] = eval_run(run)

photo sub50 42


100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:02<00:00, 20.76s/it]


photo sub50 0


100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:21<00:00, 27.14s/it]


photo sub50 66


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:19<00:00, 19.52s/it]


all sub10 12


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:14<00:00, 14.77s/it]


all sub10 123


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:09<00:00,  9.33s/it]
